# Tutorial 5: Perturbation Analysis

In-silico perturbation experiments let you predict what happens when a TF is knocked down or overexpressed. This helps validate the learned GRN and generate hypotheses for wet-lab experiments.

## Setup

In [ ]:
import torch

import deepscenic as ds

print(f"deepSCENIC version: {ds.__version__}")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## Load Model and Data

In [ ]:
data_dir = "data/MM_lines/"

# Load trained model and data
model = ds.tl.load_model(data_dir + "model.pt", device=device)
mdata = ds.read(data_dir + "preprocessed_data.h5mu")

print(f"Model: {len(model.tf_names)} TFs")

## Single TF Perturbation

Simulate TF knockdown or overexpression to predict gene expression changes:

In [ ]:
# Simulate SOX10 knockdown (complete knockout)
logFC = ds.tl.simulate_perturbation(
    model,
    mdata,
    tf_name="SOX10",
    level=0,  # 0 = complete knockdown, 2 = 2x overexpression
)

# Process results for plotting
results = ds.tl.process_perturbation_results(logFC, mdata, tf_name="SOX10")
results.head()

## Visualize Perturbation Effects

### Volcano Plot

Volcano plots show effect size vs significance:

In [ ]:
# Volcano plot shows effect size vs significance
ds.pl.volcano_perturbation(
    results,
    highlight_genes=["MITF", "DCT", "TYR"],  # Known melanocyte markers
)

```{tip}
Look for:
- **Direct targets**: Strong effect, high significance
- **Downstream effects**: Moderate effect (regulated by targets of the perturbed TF)
- **Marker genes**: Known cell-type markers should respond as expected
```

### Heatmap

Heatmap of top affected genes:

In [ ]:
# Heatmap of top affected genes
ds.pl.heatmap_perturbation(results, top_n=30)

### Dotplot by Cell Type

Dotplot showing effects by cell type (size = magnitude, color = direction):

In [ ]:
# Dotplot by cell type (if cell annotations available)
ds.pl.dotplot_perturbation(
    results,
    mdata=mdata,
    groupby="cell_state",
    genes=["MITF", "DCT", "TYR", "SOX10"],
)

## Multi-TF Perturbation

Simulate perturbation of multiple TFs simultaneously:

In [ ]:
# Simulate combined knockdown of two TFs
logFC_multi = ds.tl.simulate_multi_perturbation(
    model,
    mdata,
    tf_names=["SOX10", "MITF"],
    levels=[0, 0],  # Both knocked down
)

results_multi = ds.tl.process_perturbation_results(
    logFC_multi, mdata, tf_name=["SOX10", "MITF"]
)
results_multi.head()

## Comparing TF Effects

Compare perturbation effects across multiple TFs:

In [ ]:
# Heatmap comparing effects of different TF perturbations
ds.pl.heatmap_perturbation(results_multi, top_n=30)

## Next Steps

Now that you've analyzed perturbation effects:

- Explore advanced genomic visualizations: {doc}`06_advanced_visualization`